# Lab 7 — Large Language Models (LLMs)


Name: Nawaf Mohammed Alghamdi

ID: 2240007352


## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
%pip -q install transformers sentencepiece torch


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: /Users/nawaf/Documents/Codex/2026-11-02/referenced-chatgpt-conversation-this-is-an/work/venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


/Users/nawaf/Documents/Codex/2026-11-02/referenced-chatgpt-conversation-this-is-an/work/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1102 23:01:18.898000 49004 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


/Users/nawaf/Documents/Codex/2026-11-02/referenced-chatgpt-conversation-this-is-an/work/venv/lib/python3.11/site-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
model_families = {
    "Sentiment classification": "Encoder-only",
    "Machine translation": "Encoder–decoder",
    "Open-ended text generation": "Decoder-only",
    "Named Entity Recognition": "Encoder-only",
    "Text summarization": "Encoder–decoder",
}
for task, family in model_families.items():
    print(f"{task}: {family}")

Sentiment classification: Encoder-only
Machine translation: Encoder–decoder
Open-ended text generation: Decoder-only
Named Entity Recognition: Encoder-only
Text summarization: Encoder–decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]
predictions = classifier(sentences)
for sentence, prediction in zip(sentences, predictions):
    print(f"{sentence}\nLabel: {prediction['label']}, Score: {prediction['score']:.6f}\n")

The movie was excellent.
Label: POSITIVE, Score: 0.999861

The lecture was difficult to understand.
Label: NEGATIVE, Score: 0.999665

I enjoyed learning about transformers.
Label: POSITIVE, Score: 0.999746



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = (
    "A university library introduced a digital borrowing system this semester. "
    "Students can now reserve books online and receive reminders before their loans expire. "
    "The system has reduced waiting times at the service desk and made it easier "
    "for students to find the books they need."
)
summary_prompt = "Summarize: " + paragraph
summary_inputs = tokenizer(summary_prompt, return_tensors="pt", truncation=True)
summary_output = model.generate(
    **summary_inputs, max_new_tokens=60, num_beams=4, do_sample=False
)
print("Paragraph:", paragraph)
print("Summary:", tokenizer.decode(summary_output[0], skip_special_tokens=True))

Paragraph: A university library introduced a digital borrowing system this semester. Students can now reserve books online and receive reminders before their loans expire. The system has reduced waiting times at the service desk and made it easier for students to find the books they need.
Summary: Students can now reserve books online and receive reminders before their loans expire.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**
A decoder-only model generates a response one token at a time using the preceding conversation as context. Its causal attention lets it use earlier tokens to predict a suitable next token. This makes it well suited to producing flexible, open-ended responses and continuing multi-turn conversations. Instruction tuning and alignment further help a chatbot such as ChatGPT follow requests and provide useful answers.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [9]:
sampling_prompt = "Natural language processing is"
for attempt in range(3):
    sampled_output = generator(
        sampling_prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"Run {attempt + 1}: {sampled_output[0]['generated_text']}\n")

Run 1: Natural language processing is an important tool for developing and developing languages in a language. For example, the language processing protocol is used to generate an algorithm for the human language processing system, and it is also used to generate a



Run 2: Natural language processing is an advanced technique for learning to interpret sentences into a language. It is a technique to learn the language using a combination of three different languages. It is used to interpret sentences into a language, as well



Run 3: Natural language processing is a process of processing information and processing the data from a wide range of languages to produce and analyze data for different languages. As we are looking to change the way we parse and parse information we do not



Sampling randomly selects the next token from a probability distribution instead of always selecting the most likely token. Without resetting a fixed random seed, successive runs can select different tokens even with the same prompt. Each selected token changes the context for later predictions, so small early differences can lead to different continuations. Temperature, top-k, and top-p control the distribution used for sampling.